# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Mã nguồn dưới đây được tối ưu hóa cho Google Colab T4 (16 GB VRAM) và tích hợp sẵn GPU Compatibility Gate.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

import torch
torch.backends.cudnn.benchmark = True

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata, drive

# Mount Drive để lưu adapter và checkpoint đề phòng ngắt session
try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount notice:", e)

try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception as e:
    print("Colab secrets notice:", e)

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from scripts.labcommon import read_jsonl

train_samples = read_jsonl("data/splits/train.jsonl")
val_samples = read_jsonl("data/splits/val.jsonl")
test_samples = read_jsonl("data/splits/test.jsonl")
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

def draw_ground_truth(item):
    img = Image.open(Path("data") / item["image_path"]).convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = item["width"], item["height"]
    for obj in item["objects"]:
        xc, yc, bw, bh = obj["yolo"]
        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w
        y2 = (yc + bh / 2) * h
        draw.rectangle([x1, y1, x2, y2], outline="green", width=2)
        draw.text((x1, max(0, y1 - 10)), obj["label"], fill="green")
    return img

plt.figure(figsize=(12, 4))
for i in range(min(3, len(train_samples))):
    plt.subplot(1, 3, i + 1)
    plt.imshow(draw_ground_truth(train_samples[i]))
    plt.title(f"{train_samples[i]['image_id']}\n({train_samples[i]['session']})")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json

def yolo_to_box(label: str, yolo: list) -> dict:
    xc, yc, w, h = yolo
    x1 = int(round(max(0.0, min(1000.0, (xc - w / 2) * 1000))))
    y1 = int(round(max(0.0, min(1000.0, (yc - h / 2) * 1000))))
    x2 = int(round(max(0.0, min(1000.0, (xc + w / 2) * 1000))))
    y2 = int(round(max(0.0, min(1000.0, (yc + h / 2) * 1000))))
    if x2 <= x1:
        x2 = min(1000, x1 + 1)
    if y2 <= y1:
        y2 = min(1000, y1 + 1)
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects: list) -> str:
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

PROMPT = (
    "Locate road damage in this image. "
    "Detect only objects of classes: pothole, crack, manhole. "
    "Output a JSON list of objects with label and bbox_2d coordinates "
    "normalized on a 0-1000 scale: "
    '[{"label": "...", "bbox_2d": [x1, y1, x2, y2]}]. '
    "If no damage is found, output []."
)

# Kiểm tra tính nhất quán
sample_ans = objects_to_answer(train_samples[0]["objects"])
print("Sample JSON answer:", sample_ans)

## 3. E0: zero-shot

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from scripts.labcommon import write_jsonl
from tqdm.auto import tqdm

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"]
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

from transformers.models.qwen3_vl.modeling_qwen3_vl import Qwen3VLVisionPatchMerger
_orig_merger_forward = Qwen3VLVisionPatchMerger.forward
def _safe_merger_forward(self, x: torch.Tensor) -> torch.Tensor:
    if hasattr(self, "norm") and hasattr(self.norm, "weight") and self.norm.weight is not None:
        if x.dtype != self.norm.weight.dtype:
            x = x.to(self.norm.weight.dtype)
    return _orig_merger_forward(self, x)
Qwen3VLVisionPatchMerger.forward = _safe_merger_forward


def load_base_model():
    """Use both Kaggle T4 GPUs in one process; reserve VRAM for activations."""
    n_gpu = torch.cuda.device_count()
    opts = dict(
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
    )
    if n_gpu >= 2:
        opts["device_map"] = "balanced"
        # Leave headroom for gradients/activations/optimizer, not just model weights.
        opts["max_memory"] = {0: "11GiB", 1: "11GiB"}
    else:
        opts["device_map"] = {"": 0}
    model = Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID, **opts)
    mapping = getattr(model, "hf_device_map", {})
    print("Visible GPUs:", n_gpu, "| Model device map:", mapping)
    if n_gpu >= 2:
        devices = {str(v).replace("cuda:", "") for v in mapping.values()}
        if not {"0", "1"} <= devices:
            raise RuntimeError(
                "Both GPUs are visible but model was not split across them. "
                f"hf_device_map={mapping}. Inspect placement before training."
            )
    return model


def run_inference(model, samples, out_path, batch_size=1):
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    preds = []
    for i in tqdm(range(0, len(samples), batch_size), desc=f"Inference {Path(out_path).stem}"):
        batch = samples[i:i + batch_size]
        images = [Image.open(Path("data") / s["image_path"]).convert("RGB") for s in batch]
        messages = [
            [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": PROMPT}]}]
            for img in images
        ]
        texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages]
        inputs = processor(text=texts, images=images, return_tensors="pt", padding=True).to("cuda")
        with torch.inference_mode(), torch.amp.autocast("cuda", dtype=torch.float16):
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        prompt_len = inputs["input_ids"].shape[1]
        raw_outputs = processor.batch_decode(out[:, prompt_len:], skip_special_tokens=True)
        for s, raw in zip(batch, raw_outputs):
            preds.append({"image_id": s["image_id"], "raw": raw.strip()})
    write_jsonl(out_path, preds)

if not Path("work/e0.jsonl").exists() and not Path("report/evidence/e0.json").exists():
    model_e0 = load_base_model()
    run_inference(model_e0, test_samples, "work/e0.jsonl", batch_size=1)
    del model_e0
    torch.cuda.empty_cache()
else:
    print("E0 inference đã có sẵn, bỏ qua chạy lại.")


In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from torch.utils.data import Dataset, DataLoader

def encode_example(image, prompt, answer, proc):
    user_msg = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    full_msg = user_msg + [{"role": "assistant", "content": answer}]

    prompt_enc = proc.apply_chat_template(
        user_msg, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    )
    full_enc = proc.apply_chat_template(
        full_msg, tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors="pt"
    )

    n_prompt = prompt_enc.input_ids.shape[1]
    assert (full_enc.input_ids[0, :n_prompt] == prompt_enc.input_ids[0]).all()

    labels = full_enc.input_ids.clone()
    labels[0, :n_prompt] = -100

    assert "mm_token_type_ids" in full_enc, "Qwen3-VL processor phai tra ve mm_token_type_ids!"
    mm_ids = full_enc["mm_token_type_ids"]
    assert mm_ids.shape == full_enc.input_ids.shape, "mm_token_type_ids khong khop input_ids!"

    return {
        "input_ids": full_enc.input_ids[0],
        "attention_mask": full_enc.attention_mask[0],
        "mm_token_type_ids": mm_ids[0],
        "pixel_values": full_enc.pixel_values,
        "image_grid_thw": full_enc.image_grid_thw,
        "labels": labels[0]
    }

class RoadDamageDataset(Dataset):
    def __init__(self, samples, proc):
        self.samples = samples
        self.proc = proc

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(Path("data") / item["image_path"]).convert("RGB")
        answer = objects_to_answer(item["objects"])
        return encode_example(image, PROMPT, answer, self.proc)

def collate_fn(batch):
    pad_id = processor.tokenizer.pad_token_id or 151643
    input_ids = torch.nn.utils.rnn.pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=pad_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence([b["attention_mask"] for b in batch], batch_first=True, padding_value=0)
    labels = torch.nn.utils.rnn.pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100)
    assert all("mm_token_type_ids" in b for b in batch), "Co sample thieu mm_token_type_ids!"
    mm_token_type_ids = torch.nn.utils.rnn.pad_sequence([b["mm_token_type_ids"] for b in batch], batch_first=True, padding_value=0)
    assert mm_token_type_ids.shape == input_ids.shape, "mm_token_type_ids shape mismatch!"

    pixel_values = torch.cat([b["pixel_values"] for b in batch], dim=0)
    image_grid_thw = torch.cat([b["image_grid_thw"] for b in batch], dim=0)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "mm_token_type_ids": mm_token_type_ids,
        "labels": labels,
        "pixel_values": pixel_values,
        "image_grid_thw": image_grid_thw
    }

# Only compute LM-head logits for tokens that predict the assistant JSON.
# This avoids large logits for vision/prompt tokens whose labels are -100.
import torch.nn.functional as F

def answer_only_loss(model, batch):
    labels = batch["labels"]
    shifted = labels[:, 1:]
    supervised = shifted.ne(-100)
    if not bool(supervised.any()):
        raise ValueError("No supervised answer tokens in batch")

    # At logit index t, the model predicts labels[t + 1].
    # Keep the common tail beginning with the earliest supervised logit.
    first_logit = int(supervised.nonzero(as_tuple=False)[:, 1].min().item())
    keep = labels.shape[1] - first_logit
    inputs = {k: v for k, v in batch.items() if k != "labels"}
    out = model(**inputs, logits_to_keep=keep)
    logits = out.logits[:, :-1, :]
    target = labels[:, first_logit + 1:]
    if logits.shape[:2] != target.shape:
        raise RuntimeError(
            f"Misaligned causal loss: logits={tuple(logits.shape)}, "
            f"target={tuple(target.shape)}"
        )
    active = target.ne(-100)
    # Convert only supervised logits to fp32 for numerically stable CE.
    return F.cross_entropy(logits[active].float(), target[active])

# Lightweight mathematical regression check against the standard full-sequence
# next-token CE, with varying answer start points and right padding.
def _test_answer_loss_indexing():
    torch.manual_seed(7)
    logits = torch.randn(2, 9, 17)
    labels = torch.tensor([
        [-100, -100, -100, -100, 2, 3, 4, 5, 6],
        [-100, -100, -100, -100, -100, 7, 8, 9, -100],
    ])
    ref = F.cross_entropy(
        logits[:, :-1].reshape(-1, 17),
        labels[:, 1:].reshape(-1),
        ignore_index=-100,
    )
    positions = labels[:, 1:].ne(-100).nonzero(as_tuple=False)[:, 1]
    first = int(positions.min().item())
    tail = logits[:, first:-1]
    gt = labels[:, first + 1:]
    active = gt.ne(-100)
    actual = F.cross_entropy(tail[active], gt[active])
    assert torch.allclose(ref, actual, atol=1e-6), (ref, actual)
_test_answer_loss_indexing()
print("PASS: answer-only causal cross-entropy aligns with original loss")

# Kiem tra mask loss
chk = RoadDamageDataset(train_samples[:1], processor)[0]
masked_tokens = chk['labels'][chk['labels'] != -100]
print('Decoded assistant answer:', processor.decode(masked_tokens))


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
import re
from peft import LoraConfig, get_peft_model

# PEFT's string target_modules uses full regex matching. Resolve actual module
# paths first, then pass an exact-name list to avoid prefix/suffix surprises.
_LLM_LORA_RE = re.compile(
    r"(?:^|\.)language_model\.layers\.\d+\."
    r"(?:self_attn\.(?:q_proj|k_proj|v_proj|o_proj)"
    r"|mlp\.(?:gate_proj|up_proj|down_proj))$"
)
_VIT_LORA_RE = re.compile(
    r"(?:^|\.)visual\.blocks\.\d+\."
    r"(?:attn\.(?:qkv|qkv_proj|proj)|mlp\.(?:linear_fc1|linear_fc2))$"
)

def find_lora_targets(model, include_vit=False):
    names = [name for name, _ in model.named_modules()]
    llm = [name for name in names if _LLM_LORA_RE.search(name)]
    vit = [name for name in names if _VIT_LORA_RE.search(name)]

    # Day 10 pins Qwen3-VL-4B: 36 LLM layers x 7, 24 ViT blocks x 4.
    # Fail before PEFT injection if model/library names change.
    if len(llm) != 252 or (include_vit and len(vit) != 96):
        examples = [
            n for n in names
            if ("language_model" in n or "visual.blocks" in n)
            and (n.endswith("_proj") or n.endswith(("qkv", "qkv_proj", "linear_fc1", "linear_fc2")))
        ][:12]
        raise RuntimeError(
            f"Unexpected Qwen3-VL LoRA modules: LLM={len(llm)}/252, "
            f"ViT={len(vit)}/96; sample module names: {examples}. "
            "Inspect model.named_modules() and update the selectors; "
            "do not fall back to broad q_proj/proj suffixes."
        )

    targets = sorted(llm + (vit if include_vit else []))
    assert len(targets) == len(set(targets)), "Duplicate LoRA module targets"
    return targets

from pathlib import Path
import torch
import bitsandbytes as bnb
from safetensors.torch import save_file as save_safetensors, load_file as load_safetensors

_PROJECTOR_RE = re.compile(r"(?:^|\.)visual\.(?:merger|deepstack_merger_list)(?:\.|$)")

def is_projector_parameter(name):
    return _PROJECTOR_RE.search(name) is not None

def prepare_full_precision_projectors(model):
    """Dequantize 4-bit vision mergers *before* PEFT; casting Params4bit.data is unsafe."""
    roots = [
        n for n, _ in model.named_modules()
        if n.endswith("visual.merger") or n.endswith("visual.deepstack_merger_list")
    ]
    if not roots:
        raise RuntimeError("No vision mergers found in Qwen3-VL.")

    replaced = []
    for name, module in list(model.named_modules()):
        if not is_projector_parameter(name) or not isinstance(module, bnb.nn.Linear4bit):
            continue
        state = getattr(module.weight, "quant_state", None)
        if state is None:
            raise RuntimeError(f"{name} is Linear4bit without quant_state; cannot safely recover weights.")
        with torch.no_grad():
            weight = bnb.functional.dequantize_4bit(module.weight.data, state)
            new = torch.nn.Linear(
                module.in_features, module.out_features,
                bias=module.bias is not None, device=weight.device, dtype=torch.float32,
            )
            new.weight.copy_(weight.to(torch.float32))
            if module.bias is not None:
                new.bias.copy_(module.bias.to(torch.float32))
        model.set_submodule(name, new)
        replaced.append(name)

    # Convert other merger parameters (e.g. norms) only after removing all Linear4bit.
    for root in roots:
        model.get_submodule(root).to(dtype=torch.float32)
    assert_full_precision_projectors(model)
    print(f"Projector full precision: dequantized {len(replaced)} Linear4bit layer(s)")

def assert_full_precision_projectors(model):
    bad_modules = [
        n for n, m in model.named_modules()
        if is_projector_parameter(n) and isinstance(m, bnb.nn.Linear4bit)
    ]
    projector = [(n, p) for n, p in model.named_parameters() if is_projector_parameter(n)]
    bad_params = [
        n for n, p in projector
        if p.dtype != torch.float32 or isinstance(p, bnb.nn.Params4bit)
        or getattr(p, "quant_state", None) is not None
    ]
    if bad_modules or bad_params or not projector:
        raise RuntimeError(
            f"Projectors must be unquantized fp32; Linear4bit={bad_modules[:4]}, "
            f"invalid params={bad_params[:4]}, parameter_count={len(projector)}."
        )
    return projector

def save_projector_checkpoint(model, path):
    """Save plain CPU tensors only. Never pickle bitsandbytes Params4bit/QuantState."""
    params = assert_full_precision_projectors(model)
    state = {}
    for name, param in params:
        tensor = param.detach().to(device="cpu", dtype=torch.float32).contiguous().clone()
        if type(tensor) is not torch.Tensor:
            tensor = tensor.as_subclass(torch.Tensor).contiguous().clone()
        state[name] = tensor
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    save_safetensors(state, str(path), metadata={"format": "qwen3vl-projector-fp32"})
    return path

def load_projector_checkpoint(model, path):
    """Restore every projector parameter by exact name; never ignore missing keys."""
    params = dict(model.named_parameters())
    expected = {n for n in params if is_projector_parameter(n)}
    if not expected:
        raise RuntimeError("Model has no projector parameters to restore.")
    state = load_safetensors(str(path), device="cpu")
    if set(state) != expected:
        raise RuntimeError(
            f"Projector key mismatch: missing={sorted(expected - set(state))[:5]}, "
            f"unexpected={sorted(set(state) - expected)[:5]}"
        )
    for name, saved in state.items():
        if saved.shape != params[name].shape or not params[name].is_floating_point():
            raise RuntimeError(f"Invalid projector parameter shape/dtype: {name}")
        if isinstance(params[name], bnb.nn.Params4bit):
            raise RuntimeError(f"Cannot restore fp32 projector into quantized Params4bit: {name}")
    with torch.no_grad():
        for name, saved in state.items():
            target = params[name]
            target.copy_(saved.to(device=target.device, dtype=target.dtype))
    print(f"Restored {len(state)} projector tensors from {path}")

def apply_setup(model, setup_name: str):
    # E4_data_0.1 uses E2 architecture; only the data fraction changes.
    if setup_name not in ("E1", "E2", "E3", "E4_data_0.1"):
        raise ValueError(f"Unknown setup: {setup_name}")

    uses_projector = setup_name in ("E2", "E3", "E4_data_0.1")
    if uses_projector:
        prepare_full_precision_projectors(model)

    include_vit = setup_name == "E3"
    target_modules = find_lora_targets(model, include_vit=include_vit)
    print(f"{setup_name}: found {len(target_modules)} LoRA targets "
          f"(LLM=252, ViT={96 if include_vit else 0})")

    for param in model.parameters():
        param.requires_grad = False
    model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=target_modules, bias="none", task_type="CAUSAL_LM",
    )
    peft_model = get_peft_model(model, lora_config)
    injected = getattr(peft_model.base_model, "targeted_module_names", None)
    if injected is not None and len(injected) != len(target_modules):
        raise RuntimeError(f"PEFT injected {len(injected)}/{len(target_modules)} LoRA targets.")

    if uses_projector:
        for _, param in assert_full_precision_projectors(peft_model):
            param.requires_grad = True

    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    print(f"{setup_name} trainable params: {trainable:,}")
    return peft_model

# Cheap regression check: no model download or GPU required.
class _LoRANamesFixture:
    def named_modules(self):
        for layer in range(36):
            for name in ("self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj",
                         "self_attn.o_proj", "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"):
                yield f"model.language_model.layers.{layer}.{name}", None
        for block in range(24):
            for name in ("attn.qkv", "attn.proj", "mlp.linear_fc1", "mlp.linear_fc2"):
                yield f"model.visual.blocks.{block}.{name}", None
        yield "model.visual.merger.linear_fc1", None

assert len(find_lora_targets(_LoRANamesFixture())) == 252
assert len(find_lora_targets(_LoRANamesFixture(), include_vit=True)) == 348
print("PASS: LoRA target discovery (252 LLM / 96 ViT)")


In [ ]:
# === GPU COMPATIBILITY GATE (SMOKE TEST) ===
print("--- RUNNING GPU COMPATIBILITY GATE ---")
from torch.amp import autocast

smoke_base = load_base_model()
smoke_base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
smoke_base.config.use_cache = False
smoke_model = apply_setup(smoke_base, "E2")
for n, p in smoke_model.named_parameters():
    if "visual.merger" in n:
        assert p.dtype == torch.float32, f"{n} phai o float32!"
        assert p.requires_grad, f"{n} phai co requires_grad=True!"

smoke_loader = DataLoader(
    RoadDamageDataset(train_samples[:1], processor),
    batch_size=1,
    collate_fn=collate_fn
)
smoke_batch = next(iter(smoke_loader))
assert "mm_token_type_ids" in smoke_batch, "smoke_batch thieu mm_token_type_ids!"
assert smoke_batch["mm_token_type_ids"].shape == smoke_batch["input_ids"].shape, "Shape mm_token_type_ids khong khop!"
print("Batch keys:", list(smoke_batch.keys()))

smoke_batch = {k: v.to("cuda") for k, v in smoke_batch.items()}
with autocast("cuda", dtype=torch.float16):
    smoke_loss = answer_only_loss(smoke_model, smoke_batch)

assert torch.isfinite(smoke_loss), "Loss bi NaN/Inf!"
smoke_loss.backward()

assert any(p.grad is not None and torch.isfinite(p.grad).all() for n, p in smoke_model.named_parameters() if "visual.merger" in n), "Projector khong nhan gradient!"
smoke_save_path = save_projector_checkpoint(smoke_model, "work/smoke_projector.safetensors")
saved_projector = load_safetensors(str(smoke_save_path), device="cpu")
projector_params = dict(assert_full_precision_projectors(smoke_model))
assert set(saved_projector) == set(projector_params)
assert all(
    torch.equal(saved_projector[n], p.detach().cpu())
    for n, p in projector_params.items()
), "Projector save/load round-trip mismatch"
del saved_projector, projector_params
del smoke_model, smoke_base, smoke_loss, smoke_batch
import gc
gc.collect()
torch.cuda.empty_cache()
print("Peak GPU memory (GiB):", {
    str(i): round(torch.cuda.max_memory_allocated(i) / 2**30, 2)
    for i in range(torch.cuda.device_count())
})
print(">>> GPU COMPATIBILITY GATE: PASS <<<")


## 6. Vòng huấn luyện

In [ ]:
import time
import math
from torch.amp import GradScaler, autocast

def train_model(setup_name, samples_train, max_steps=300, lr=2e-4, batch_size=1, grad_accum=8):
    meta_file = Path(f"work/{setup_name}_meta.json")
    adapter_dir = Path(f"work/{setup_name}_adapter")
    proj_file = Path(f"work/{setup_name}_projector.safetensors")
    needs_proj = setup_name in ("E2", "E3", "E4_data_0.1")
    has_adapter = adapter_dir.exists() and (adapter_dir / "adapter_config.json").exists()
    has_proj = (not needs_proj) or proj_file.exists()

    if meta_file.exists() and has_adapter and has_proj:
        print(f"=== Checkpoint cho {setup_name} đã tồn tại trong work/. Bỏ qua train để chạy nhanh hơn! ===")
        return json.loads(meta_file.read_text())
    for dev in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(dev)
    start_time = time.time()

    base_model = load_base_model()
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base_model.config.use_cache = False

    model = apply_setup(base_model, setup_name)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Tối ưu DataLoader: num_workers=2, pin_memory=True, persistent_workers=True
    train_loader = DataLoader(
        RoadDamageDataset(samples_train, processor),
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True
    )

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.0)
    scaler = GradScaler("cuda")

    warmup_steps = int(max_steps * 0.05)
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, max_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    model.train()
    step = 0
    pbar = tqdm(total=max_steps, desc=f"Training {setup_name}")
    epoch_iter = iter(train_loader)

    while step < max_steps:
        optimizer.zero_grad()
        for _ in range(grad_accum):
            try:
                batch = next(epoch_iter)
            except StopIteration:
                epoch_iter = iter(train_loader)
                batch = next(epoch_iter)

            batch = {k: v.to("cuda", non_blocking=True) for k, v in batch.items()}
            with autocast("cuda", dtype=torch.float16):
                loss = answer_only_loss(model, batch) / grad_accum

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite loss encountered!")
            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        step += 1
        pbar.update(1)
        pbar.set_postfix({"loss": f"{loss.item() * grad_accum:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
    pbar.close()

    elapsed_mins = (time.time() - start_time) / 60.0
    peak_by_gpu = {
        str(dev): round(torch.cuda.max_memory_allocated(dev) / (1024 ** 3), 2)
        for dev in range(torch.cuda.device_count())
    }
    peak_vram = max(peak_by_gpu.values())
    meta = {
        "trainable_params": trainable_params,
        "peak_vram_gb": peak_vram,
        "peak_vram_per_gpu_gb": peak_by_gpu,
        "train_minutes": round(elapsed_mins, 2)
    }

    adapter_dir = f"work/{setup_name}_adapter"
    model.save_pretrained(adapter_dir)
    if setup_name in ("E2", "E3", "E4_data_0.1"):
        save_projector_checkpoint(model, f"work/{setup_name}_projector.safetensors")

    Path(f"work/{setup_name}_meta.json").write_text(json.dumps(meta, indent=2))
    print(f"{setup_name} completed in {meta['train_minutes']} min | Peak VRAM: {meta['peak_vram_gb']} GB")

    del model, base_model, optimizer, train_loader
    torch.cuda.empty_cache()
    return meta


## 7. Suy luận và chấm E1–E3

In [ ]:
from peft import PeftModel
import subprocess

def evaluate_and_score(setup_name):
    evidence_path = Path(f"report/evidence/{setup_name.lower()}.json")
    if evidence_path.exists():
        print(f"=== Evidence {evidence_path} đã tồn tại. Bỏ qua evaluate_and_score cho {setup_name}! ===")
        return
    base = load_base_model()
    if setup_name in ("E2", "E3"):
        prepare_full_precision_projectors(base)
    model = PeftModel.from_pretrained(base, f"work/{setup_name}_adapter")

    if setup_name in ("E2", "E3"):
        proj_path = Path(f"work/{setup_name}_projector.safetensors")
        if not proj_path.exists():
            raise FileNotFoundError(f"Missing trained projector checkpoint: {proj_path}")
        load_projector_checkpoint(model, proj_path)

    model.eval()
    pred_path = f"work/{setup_name.lower()}.jsonl"
    run_inference(model, test_samples, pred_path, batch_size=1)

    del model, base
    torch.cuda.empty_cache()

    meta = json.loads(Path(f"work/{setup_name}_meta.json").read_text())
    cmd = [
        "python", "scripts/score.py",
        "--pred", pred_path,
        "--name", setup_name,
        "--out", str(evidence_path),
        "--meta", f"trainable_params={meta['trainable_params']}",
        "--meta", f"peak_vram_gb={meta['peak_vram_gb']}",
        "--meta", f"train_minutes={meta['train_minutes']}"
    ]
    subprocess.run(cmd, check=True)

# Chạy tuần tự E1, E2, E3
for s in ["E1", "E2", "E3"]:
    print(f"=== Bắt đầu {s} ===")
    train_model(s, train_samples)
    evaluate_and_score(s)


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# === E4: Ablation study trên data_size (10% vs 100%) ===
import random
random.seed(42)

e4_evidence = Path("report/evidence/e4_data_0.1.json")
if not e4_evidence.exists():
    n_10pct = max(1, int(len(train_samples) * 0.1))
    train_10pct = random.sample(train_samples, n_10pct)

    print("=== Bắt đầu E4 (10% data) ===")
    train_model("E4_data_0.1", train_10pct)

    # Suy luận và chấm điểm E4
    base_e4 = load_base_model()
    prepare_full_precision_projectors(base_e4)
    model_e4 = PeftModel.from_pretrained(base_e4, "work/E4_data_0.1_adapter")
    load_projector_checkpoint(model_e4, "work/E4_data_0.1_projector.safetensors")

    run_inference(model_e4, test_samples, "work/e4_data_0.1.jsonl", batch_size=1)
    del model_e4, base_e4
    torch.cuda.empty_cache()

    meta_e4 = json.loads(Path("work/E4_data_0.1_meta.json").read_text())
    subprocess.run([
        "python", "scripts/score.py",
        "--pred", "work/e4_data_0.1.jsonl",
        "--name", "E4_data_0.1",
        "--out", str(e4_evidence),
        "--meta", f"trainable_params={meta_e4['trainable_params']}",
        "--meta", f"peak_vram_gb={meta_e4['peak_vram_gb']}",
        "--meta", f"train_minutes={meta_e4['train_minutes']}"
    ], check=True)
else:
    print(f"=== Evidence {e4_evidence} đã tồn tại. Bỏ qua E4! ===")


In [ ]:
# === E5 (Tuỳ chọn): GRPO với IoU Reward và Unit Test ===
from scripts.score import parse_output, iou_xyxy

def compute_reward(completion: str, gt_boxes: list) -> float:
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not gt_boxes and not pred:
        return 1.0

    total_iou = 0.0
    all_labels = {g["label"] for g in gt_boxes} | {p["label"] for p in pred}
    for lab in all_labels:
        g = [x["bbox"] for x in gt_boxes if x["label"] == lab]
        p = [x["bbox"] for x in pred if x["label"] == lab]
        pairs = sorted(((iou_xyxy(a, b), i, j) for i, a in enumerate(g) for j, b in enumerate(p)), reverse=True)
        used_g, used_p = set(), set()
        for v, i, j in pairs:
            if v < 0.5:
                break
            if i in used_g or j in used_p:
                continue
            used_g.add(i)
            used_p.add(j)
            total_iou += v

    denom = max(len(gt_boxes), len(pred), 1)
    return 0.2 + 0.8 * (total_iou / denom)

# Kiểm tra 5 ca test của hàm reward
gt_demo = [{"label": "pothole", "bbox": (100.0, 100.0, 200.0, 200.0)}]
assert compute_reward(json.dumps([{"label": "pothole", "bbox_2d": [100, 100, 200, 200]}]), gt_demo) == 1.0
assert compute_reward("invalid json", gt_demo) == 0.0
assert compute_reward(json.dumps([{"label": "crack", "bbox_2d": [100, 100, 200, 200]}]), gt_demo) == 0.2
assert compute_reward(json.dumps([{"label": "pothole", "bbox_2d": [100, 100, 200, 200]}, {"label": "pothole", "bbox_2d": [10, 10, 20, 20]}]), gt_demo) == 0.6
assert compute_reward("[]", []) == 1.0
print("Reward IoU unit tests: 5/5 PASSED")

In [ ]:
# === Trích xuất tối thiểu 6 ảnh minh hoạ Ground Truth vs Prediction ===
from scripts.score import read_jsonl

eval_pred_path = "work/e2.jsonl" if Path("work/e2.jsonl").exists() else "work/e1.jsonl"
if Path(eval_pred_path).exists():
    preds_dict = {r["image_id"]: parse_output(r["raw"]) for r in read_jsonl(eval_pred_path)}
    report_dir = Path("report")
    report_dir.mkdir(parents=True, exist_ok=True)
    test_dict = {t["image_id"]: t for t in test_samples}
    sample_ids = list(test_dict.keys())[:6]

    for idx, img_id in enumerate(sample_ids):
        item = test_dict[img_id]
        img = Image.open(Path("data") / item["image_path"]).convert("RGB")
        draw = ImageDraw.Draw(img)
        w, h = item["width"], item["height"]

        for obj in item["objects"]:
            xc, yc, bw, bh = obj["yolo"]
            x1, y1 = (xc - bw / 2) * w, (yc - bh / 2) * h
            x2, y2 = (xc + bw / 2) * w, (yc + bh / 2) * h
            draw.rectangle([x1, y1, x2, y2], outline="green", width=3)
            draw.text((x1, max(0, y1 - 12)), f"GT: {obj['label']}", fill="green")

        preds = preds_dict.get(img_id) or []
        for p in preds:
            bx1, by1, bx2, by2 = p["bbox"]
            x1, y1 = bx1 / 1000 * w, by1 / 1000 * h
            x2, y2 = bx2 / 1000 * w, by2 / 1000 * h
            draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
            draw.text((x1, min(h - 12, y2 + 2)), f"Pred: {p['label']}", fill="red")

        out_img = report_dir / f"sample_vis_{idx+1}.jpg"
        img.save(out_img)
    print(f"Đã lưu 6 ảnh minh hoạ vào {report_dir}")
else:
    print("Chưa có file predictions để trực quan hoá.")

In [ ]:
# Kiểm tra tính hợp lệ bài nộp
!python scripts/check_submission.py